# <a id='toc1_'></a>[How adherent is the medial literature to HGNC recommendations on the use of gene identifiers?](#toc0_)

This analysis assesses how closely the medical literature follows HGNC recommendations for unambiguous gene identification. Using PubTator 3 annotations, the code identifies papers(titles, abstracts, or full text when available) in which an alias is tagged as a gene and determines whether the mention is linked to an HGNC, Ensembl, or NCBI Gene identifier. The resulting proportion provides a measure of identifier use for an alias symbol in published literature.

**Table of contents**<a id='toc0_'></a>    
- [How adherent is the medial literature to HGNC recommendations on the use of gene identifiers?](#toc1_)    
  - [Gene Pairs](#toc1_1_)    
  - [Download cache for ERBB and ASP gene pairs](#toc1_2_)    
  - [Results](#toc1_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import shutil
from collections import defaultdict
from pathlib import Path

import functions as fn


## <a id='toc1_1_'></a>[Gene Pairs](#toc0_)

In [3]:
ERBB_EGFR = fn.GenePair(
    alias="ERBB",
    approved_symbol="EGFR",
    ncbi_gene_id="1956",
    hgnc_id="HGNC:3236",
    ensembl_gene_id="ENSG00000146648",
)

ASP_TMPRSS11D = fn.GenePair(
    alias="ASP",
    approved_symbol="TMPRSS11D",
    ncbi_gene_id="9407",
    hgnc_id="HGNC:24059",
    ensembl_gene_id="ENSG00000153802",
)

ASP_C3 = fn.GenePair(
    alias="ASP",
    approved_symbol="C3",
    ncbi_gene_id="718",
    hgnc_id="HGNC:1318",
    ensembl_gene_id="ENSG00000125730",
)

ASP_ASPM = fn.GenePair(
    alias="ASP",
    approved_symbol="ASPM",
    ncbi_gene_id="259266",
    hgnc_id="HGNC:19048",
    ensembl_gene_id="ENSG00000066279",
)

ASP_A1CF = fn.GenePair(
    alias="ASP",
    approved_symbol="A1CF",
    ncbi_gene_id="29974",
    hgnc_id="HGNC:24086",
    ensembl_gene_id="ENSG00000148584",
)

ASP_ASIP = fn.GenePair(
    alias="ASP",
    approved_symbol="ASIP",
    ncbi_gene_id="434",
    hgnc_id="HGNC:745",
    ensembl_gene_id="ENSG00000101440",
)

ASP_ROPN1L = fn.GenePair(
    alias="ASP",
    approved_symbol="ROPN1L",
    ncbi_gene_id="83853",
    hgnc_id="HGNC:24060",
    ensembl_gene_id="ENSG00000145491",
)

ASP_ATG5 = fn.GenePair(
    alias="ASP",
    approved_symbol="ATG5",
    ncbi_gene_id="9474",
    hgnc_id="HGNC:589",
    ensembl_gene_id="ENSG00000057663",
)

ASP_ASPA = fn.GenePair(
    alias="ASP",
    approved_symbol="ASPA",
    ncbi_gene_id="443",
    hgnc_id="HGNC:756",
    ensembl_gene_id="ENSG00000108381",
)


## <a id='toc1_2_'></a>[Download cache for ERBB and ASP gene pairs](#toc0_)

In [4]:
CACHE_URL = "https://nch-igm-wagner-lab-public.s3.us-east-2.amazonaws.com/genejar/output/"

In [5]:
ASP_URL = f"{CACHE_URL}asp"
ERBB_URL = f"{CACHE_URL}erbb"

In [6]:
cache_dir = Path("output/erbb/document_cache")
zip_path = Path("output/erbb/document_cache.zip")

if not cache_dir.exists() or not any(cache_dir.iterdir()):
    cache_dir.parent.mkdir(parents=True, exist_ok=True)

    fn.download_s3(
        f"{ERBB_URL}/document_cache.zip",
        zip_path,
    )

    shutil.unpack_archive(
        zip_path,
        cache_dir.parent,
    )

    zip_path.unlink()

In [7]:
cache_dir = Path("output/asp/document_cache")
zip_path = Path("output/asp/document_cache.zip")

if not cache_dir.exists() or not any(cache_dir.iterdir()):
    cache_dir.parent.mkdir(parents=True, exist_ok=True)

    fn.download_s3(
        f"{ASP_URL}/document_cache.zip",
        zip_path,
    )

    shutil.unpack_archive(
        zip_path,
        cache_dir.parent,
    )

    zip_path.unlink()

## <a id='toc1_3_'></a>[Results](#toc0_)

**Candidate papers for alias:** This is the number of unique PubMed articles returned by the PubTator search for the query "alias". It comes from load_or_search_pmids(pair), which searches PubTator and caches the resulting PMIDs.

**Documents retrieved:** This is the number of PubTator BioC documents successfully downloaded from the export endpoint. It is incremented once for every document returned by fetch_documents(). If it is fewer than the candidate papers it means PubTator only had exportable BioC records for these.

**Papers where PubTator tagged the alias as a gene:** Papers with at least one PubTator annotation where:

- annotation["infons"]["type"] == "gene"
- annotation["text"] == "alias" (case-insensitive)

Each paper is counted only once, even if the alias appears multiple times.

**Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for the alias:** Papers with one of the following identifier strings in the PubTator text:

- HGNC:XXXX
- ENSG00000XXXXXX
- NCBI Gene XXXX (or equivalent pattern)

This is based on searching the title/abstract text with the regular expressions in make_identifier_patterns().

**Percentage containing an identifier:** numerator / denominator

**Counts by identifier namespace**

These are the numerator broken down by identifier type:

- NCBI Gene: papers containing an NCBI Gene identifier for the alias
- HGNC: papers containing an HGNC identifier for the alias
- Ensembl: papers containing an Ensembl identifier for the alias

In [8]:
candidate_pmids = fn.load_or_search_pmids(ERBB_EGFR)

results = fn.analyze_gene_pair(
    ERBB_EGFR,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ERBB: 25,467
Documents retrieved: 25,464
Papers where PubTator tagged the exact alias ERBB as a gene: 13,290
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for EGFR: 3
Percentage containing an identifier: 0.02%

Counts by identifier namespace:
  NCBI Gene: 1 papers
  HGNC: 0 papers
  Ensembl: 2 papers

Identifier locations (PMIDs):
  fig_caption: 32210363
  paragraph: 26886748
  table: 33232279


In [9]:
results["papers_with_identifier_in_text"]

{'26886748', '32210363', '33232279'}

In [10]:
abstract_only, full_text = fn.count_document_types(
    ERBB_EGFR,
    candidate_pmids,
)

print(f"Abstract-only documents: {abstract_only:,}")
print(f"Documents with full text: {full_text:,}")

Abstract-only documents: 7,297
Documents with full text: 18,167


In [11]:
candidate_pmids = fn.load_or_search_pmids(ASP_TMPRSS11D)

results = fn.analyze_gene_pair(
    ASP_TMPRSS11D,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Documents retrieved: 6,548
Papers where PubTator tagged the exact alias ASP as a gene: 760
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for TMPRSS11D: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [12]:
abstract_only, full_text = fn.count_document_types(
    ASP_TMPRSS11D,
    candidate_pmids,
)

print(f"Abstract-only documents: {abstract_only:,}")
print(f"Documents with full text: {full_text:,}")

Abstract-only documents: 3,681
Documents with full text: 2,867


In [13]:
candidate_pmids = fn.load_or_search_pmids(ASP_C3)

results = fn.analyze_gene_pair(
    ASP_C3,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Documents retrieved: 6,548
Papers where PubTator tagged the exact alias ASP as a gene: 760
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for C3: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [14]:
candidate_pmids = fn.load_or_search_pmids(ASP_ASPM)

results = fn.analyze_gene_pair(
    ASP_ASPM,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Documents retrieved: 6,548
Papers where PubTator tagged the exact alias ASP as a gene: 760
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ASPM: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [15]:
candidate_pmids = fn.load_or_search_pmids(ASP_A1CF)

results = fn.analyze_gene_pair(
    ASP_A1CF,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Documents retrieved: 6,548
Papers where PubTator tagged the exact alias ASP as a gene: 760
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for A1CF: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [16]:
candidate_pmids = fn.load_or_search_pmids(ASP_ASIP)

results = fn.analyze_gene_pair(
    ASP_ASIP,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Documents retrieved: 6,548
Papers where PubTator tagged the exact alias ASP as a gene: 760
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ASIP: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [17]:
candidate_pmids = fn.load_or_search_pmids(ASP_ASPA)

results = fn.analyze_gene_pair(
    ASP_ASPA,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Documents retrieved: 6,548
Papers where PubTator tagged the exact alias ASP as a gene: 760
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ASPA: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [18]:
candidate_pmids = fn.load_or_search_pmids(ASP_ATG5)

results = fn.analyze_gene_pair(
    ASP_ATG5,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Documents retrieved: 6,548
Papers where PubTator tagged the exact alias ASP as a gene: 760
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ATG5: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [19]:
candidate_pmids = fn.load_or_search_pmids(ASP_ROPN1L)

results = fn.analyze_gene_pair(
    ASP_ROPN1L,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Documents retrieved: 6,548
Papers where PubTator tagged the exact alias ASP as a gene: 760
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ROPN1L: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


Pubtator3 normalizes ASP to different genes, there are just no gene identifiers used

In [20]:
papers_by_pubtator_id = defaultdict(set)
annotations_by_pubtator_id = defaultdict(int)
papers_with_no_id = set()

for document in fn.fetch_documents(ASP_ROPN1L, candidate_pmids):
    pmid = str(
        document.get("pmid")
        or document.get("id")
        or ""
    ).strip()

    if not pmid:
        continue

    for passage in document.get("passages", []):
        for annotation in passage.get("annotations", []):
            infons = annotation.get("infons", {})

            entity_type = (
                str(infons.get("type", ""))
                .strip()
                .casefold()
            )

            mention = (
                str(annotation.get("text", ""))
                .strip()
            )

            if (
                entity_type == "gene"
                and mention.casefold() == ASP_ROPN1L.alias.casefold()
            ):
                identifier = infons.get("identifier")

                if identifier:
                    identifier = str(identifier).strip()

                    papers_by_pubtator_id[identifier].add(pmid)
                    annotations_by_pubtator_id[identifier] += 1
                else:
                    papers_with_no_id.add(pmid)

print(f"PubTator normalization results for {ASP_ROPN1L.alias}:\n")

for identifier in sorted(
    papers_by_pubtator_id,
    key=lambda x: len(papers_by_pubtator_id[x]),
    reverse=True,
):
    print(
        f"{identifier}: "
        f"{len(papers_by_pubtator_id[identifier]):,} papers, "
        f"{annotations_by_pubtator_id[identifier]:,} annotations"
    )

print(
    f"\nNo PubTator identifier: "
    f"{len(papers_with_no_id):,} papers"
)

PubTator normalization results for ASP:

259266: 726 papers, 25,529 annotations
2200: 12 papers, 286 annotations
509432: 10 papers, 236 annotations
374569: 4 papers, 94 annotations
434: 3 papers, 64 annotations
492296: 2 papers, 36 annotations
112935892: 1 papers, 3 annotations
104816: 1 papers, 24 annotations
492297: 1 papers, 10 annotations
42946: 1 papers, 1 annotations
100127221: 1 papers, 1 annotations

No PubTator identifier: 0 papers
